In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt


DATE_KEYWORDS = [
    "date",
    "time",
    "created",
    "ordered",
    "order_date",
    "invoice_date",
    "purchase_date",
    "transaction_date",
]

AMOUNT_KEYWORDS = [
    "amount",
    "price",
    "total",
    "revenue",
    "sales",
    "cost",
    "expense",
    "value",
    "payment",
    "subtotal",
    "profit",
]

CATEGORY_KEYWORDS = [
    "category",
    "type",
    "status",
    "segment",
    "product",
    "department",
    "country",
    "city",
    "region",
    "customer",
]


def clean_column_name(column_name):
    """
    Convert column names into clean snake_case format.

    Example:
    "Order Date " -> "order_date"
    "Customer-Name" -> "customer_name"
    """
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = column_name.strip("_")

    if column_name == "":
        return "unknown_column"

    return column_name


def make_unique_column_names(columns):
    """
    Avoid duplicated column names after cleaning.

    Example:
    ["total price", "total-price"] could both become "total_price".
    This function turns them into:
    ["total_price", "total_price_2"]
    """
    used_names = {}
    unique_columns = []

    for column in columns:
        clean_name = clean_column_name(column)

        if clean_name not in used_names:
            used_names[clean_name] = 1
            unique_columns.append(clean_name)
        else:
            used_names[clean_name] += 1
            unique_columns.append(f"{clean_name}_{used_names[clean_name]}")

    return unique_columns


def read_csv_safely(file_path):
    """
    Read a CSV file.

    First, we try UTF-8.
    If that fails, we try latin1 because some older CSV files use it.
    """
    try:
        return pd.read_csv(file_path)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding="latin1")


def convert_text_to_number(value):
    """
    Convert messy money/number text into a real number.

    Examples:
    "€1,200.50" -> 1200.50
    "1.200,50"  -> 1200.50
    " 450 "     -> 450
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if value == "":
        return pd.NA

    # Remove currency symbols, letters, and spaces.
    value = re.sub(r"[^\d,.\-]", "", value)

    # If both comma and dot exist, decide which one is decimal separator.
    if "," in value and "." in value:
        if value.rfind(",") > value.rfind("."):
            # European style: 1.200,50
            value = value.replace(".", "")
            value = value.replace(",", ".")
        else:
            # US/UK style: 1,200.50
            value = value.replace(",", "")

    # If only comma exists, it might be a decimal comma.
    elif "," in value:
        if re.search(r",\d{1,2}$", value):
            value = value.replace(",", ".")
        else:
            value = value.replace(",", "")

    try:
        return float(value)
    except ValueError:
        return pd.NA


def parse_dates_safely(series):
    """
    Convert a column into dates.

    format='mixed' helps pandas work with different date formats.
    dayfirst=True is useful for European dates like 31.01.2025.
    ISO dates like 2025-01-31 are still handled correctly.
    """
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )


def detect_numeric_columns(df, minimum_success_rate=0.70):
    """
    Detect columns that can probably be treated as numbers.

    We do not rely only on the column name.
    We also test if most values can actually become numbers.
    """
    numeric_columns = []

    for column in df.columns:
        converted = df[column].apply(convert_text_to_number)
        success_rate = converted.notna().mean()

        name_suggests_number = any(keyword in column for keyword in AMOUNT_KEYWORDS)

        if success_rate >= minimum_success_rate or (
            name_suggests_number and success_rate >= 0.40
        ):
            numeric_columns.append(column)

    return numeric_columns


def detect_date_columns(df, minimum_success_rate=0.70):
    """
    Detect columns that can probably be treated as dates.

    Invalid dates become NaT instead of crashing the program.
    """
    date_columns = []

    for column in df.columns:
        name_suggests_date = any(keyword in column for keyword in DATE_KEYWORDS)

        # Avoid trying to parse columns that are already clearly numeric.
        if pd.api.types.is_numeric_dtype(df[column]):
            continue

        parsed_dates = parse_dates_safely(df[column])
        success_rate = parsed_dates.notna().mean()

        if success_rate >= minimum_success_rate or (
            name_suggests_date and success_rate >= 0.40
        ):
            date_columns.append(column)

    return date_columns


def choose_main_column(columns, keywords):
    """
    Choose the most important column from a list.

    Example:
    If we have ["unit_price", "quantity", "total_amount"],
    this function prefers "total_amount" because it matches useful keywords.
    """
    for keyword in keywords:
        for column in columns:
            if keyword in column:
                return column

    if columns:
        return columns[0]

    return None


def detect_category_columns(df, max_unique_values=30):
    """
    Detect useful category columns.

    A category column usually has repeated text values:
    product category, region, status, customer segment, etc.
    """
    category_columns = []

    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            continue

        unique_count = df[column].nunique(dropna=True)
        row_count = len(df)

        if row_count == 0:
            continue

        unique_ratio = unique_count / row_count
        name_suggests_category = any(keyword in column for keyword in CATEGORY_KEYWORDS)

        if name_suggests_category or (
            2 <= unique_count <= max_unique_values and unique_ratio <= 0.50
        ):
            category_columns.append(column)

    return category_columns


def clean_dataframe(df):
    """
    Main cleaning function.

    It returns:
    1. cleaned DataFrame
    2. cleaning log with important numbers
    3. detected column groups
    """
    cleaning_log = {}

    cleaning_log["original_rows"] = len(df)
    cleaning_log["original_columns"] = len(df.columns)

    # Make a copy so we do not accidentally change the original DataFrame.
    df = df.copy()

    # Clean column names.
    df.columns = make_unique_column_names(df.columns)

    # Drop completely empty rows.
    before_empty_drop = len(df)
    df = df.dropna(how="all")
    cleaning_log["empty_rows_removed"] = before_empty_drop - len(df)

    # Strip extra spaces from text columns.
    text_columns = df.select_dtypes(include=["object"]).columns

    for column in text_columns:
        df[column] = df[column].astype(str).str.strip()
        df[column] = df[column].replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "NONE": pd.NA,
                "null": pd.NA,
                "NULL": pd.NA,
            }
        )

    # Remove duplicate rows.
    before_duplicates = len(df)
    df = df.drop_duplicates(ignore_index=True)
    cleaning_log["duplicate_rows_removed"] = before_duplicates - len(df)

    # Detect and convert date columns.
    date_columns = detect_date_columns(df)

    for column in date_columns:
        df[column] = parse_dates_safely(df[column])

    # Detect and convert numeric columns.
    numeric_columns = detect_numeric_columns(df)

    for column in numeric_columns:
        df[column] = df[column].apply(convert_text_to_number)
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Detect category columns after numeric conversion.
    category_columns = detect_category_columns(df)

    cleaning_log["cleaned_rows"] = len(df)
    cleaning_log["cleaned_columns"] = len(df.columns)
    cleaning_log["total_missing_cells"] = int(df.isna().sum().sum())

    detected_columns = {
        "date_columns": date_columns,
        "numeric_columns": numeric_columns,
        "category_columns": category_columns,
    }

    return df, cleaning_log, detected_columns


def create_report(df, cleaning_log, detected_columns, output_folder):
    """
    Create report.csv with key metrics.

    The report has a simple structure:
    metric | value
    """
    report_rows = []

    for metric, value in cleaning_log.items():
        report_rows.append(
            {
                "metric": metric,
                "value": value,
            }
        )

    report_rows.append(
        {
            "metric": "detected_date_columns",
            "value": ", ".join(detected_columns["date_columns"]) or "None",
        }
    )

    report_rows.append(
        {
            "metric": "detected_numeric_columns",
            "value": ", ".join(detected_columns["numeric_columns"]) or "None",
        }
    )

    report_rows.append(
        {
            "metric": "detected_category_columns",
            "value": ", ".join(detected_columns["category_columns"]) or "None",
        }
    )

    # Missing values by column.
    for column in df.columns:
        report_rows.append(
            {
                "metric": f"missing_values__{column}",
                "value": int(df[column].isna().sum()),
            }
        )

    # Numeric summaries.
    for column in detected_columns["numeric_columns"]:
        report_rows.extend(
            [
                {"metric": f"sum__{column}", "value": round(df[column].sum(), 2)},
                {"metric": f"average__{column}", "value": round(df[column].mean(), 2)},
                {"metric": f"min__{column}", "value": round(df[column].min(), 2)},
                {"metric": f"max__{column}", "value": round(df[column].max(), 2)},
            ]
        )

    # Date summaries.
    for column in detected_columns["date_columns"]:
        report_rows.extend(
            [
                {"metric": f"first_date__{column}", "value": df[column].min()},
                {"metric": f"last_date__{column}", "value": df[column].max()},
            ]
        )

    # Category summaries.
    for column in detected_columns["category_columns"]:
        top_values = df[column].value_counts(dropna=True)

        if not top_values.empty:
            report_rows.extend(
                [
                    {
                        "metric": f"unique_values__{column}",
                        "value": int(df[column].nunique(dropna=True)),
                    },
                    {
                        "metric": f"top_value__{column}",
                        "value": top_values.index[0],
                    },
                    {
                        "metric": f"top_value_count__{column}",
                        "value": int(top_values.iloc[0]),
                    },
                ]
            )

    report_df = pd.DataFrame(report_rows)
    report_path = output_folder / "report.csv"
    report_df.to_csv(report_path, index=False)

    return report_path


def create_extra_tables(df, detected_columns, output_folder):
    """
    Create extra useful CSV files if the needed columns exist.

    Example:
    - monthly_report.csv if we have a date column and a money column
    - category_report.csv if we have a category column and a money column
    """
    amount_column = choose_main_column(
        detected_columns["numeric_columns"],
        AMOUNT_KEYWORDS,
    )

    date_column = choose_main_column(
        detected_columns["date_columns"],
        DATE_KEYWORDS,
    )

    category_column = choose_main_column(
        detected_columns["category_columns"],
        CATEGORY_KEYWORDS,
    )

    if amount_column and date_column:
        monthly_report = (
            df.dropna(subset=[date_column])
            .assign(month=df[date_column].dt.to_period("M").astype(str))
            .groupby("month", as_index=False)[amount_column]
            .sum()
            .rename(columns={amount_column: f"total_{amount_column}"})
        )

        monthly_report.to_csv(output_folder / "monthly_report.csv", index=False)

    if amount_column and category_column:
        category_report = (
            df.groupby(category_column, dropna=False)[amount_column]
            .sum()
            .sort_values(ascending=False)
            .reset_index()
            .rename(columns={amount_column: f"total_{amount_column}"})
        )

        category_report.to_csv(output_folder / "category_report.csv", index=False)


def create_charts(df, detected_columns, output_folder):
    """
    Create simple charts and save them as PNG files.
    """
    chart_folder = output_folder / "charts"
    chart_folder.mkdir(parents=True, exist_ok=True)

    created_charts = []

    # Chart 1: missing values by column.
    missing_values = df.isna().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

    if not missing_values.empty:
        plt.figure(figsize=(10, 5))
        missing_values.plot(kind="bar")
        plt.title("Missing Values by Column")
        plt.xlabel("Column")
        plt.ylabel("Missing values")
        plt.tight_layout()

        chart_path = chart_folder / "missing_values.png"
        plt.savefig(chart_path)
        plt.close()

        created_charts.append(chart_path)

    amount_column = choose_main_column(
        detected_columns["numeric_columns"],
        AMOUNT_KEYWORDS,
    )

    date_column = choose_main_column(
        detected_columns["date_columns"],
        DATE_KEYWORDS,
    )

    category_column = choose_main_column(
        detected_columns["category_columns"],
        CATEGORY_KEYWORDS,
    )

    # Chart 2: distribution of the main numeric column.
    if amount_column:
        plt.figure(figsize=(8, 5))
        df[amount_column].dropna().plot(kind="hist", bins=20)
        plt.title(f"Distribution of {amount_column}")
        plt.xlabel(amount_column)
        plt.ylabel("Frequency")
        plt.tight_layout()

        chart_path = chart_folder / f"distribution_{amount_column}.png"
        plt.savefig(chart_path)
        plt.close()

        created_charts.append(chart_path)

    # Chart 3: monthly total if we have date + numeric column.
    if amount_column and date_column:
        monthly_data = (
            df.dropna(subset=[date_column])
            .assign(month=df[date_column].dt.to_period("M").astype(str))
            .groupby("month")[amount_column]
            .sum()
        )

        if not monthly_data.empty:
            plt.figure(figsize=(10, 5))
            monthly_data.plot(kind="line", marker="o")
            plt.title(f"Monthly Total: {amount_column}")
            plt.xlabel("Month")
            plt.ylabel(f"Total {amount_column}")
            plt.xticks(rotation=45)
            plt.tight_layout()

            chart_path = chart_folder / f"monthly_total_{amount_column}.png"
            plt.savefig(chart_path)
            plt.close()

            created_charts.append(chart_path)

    # Chart 4: top categories if we have category + numeric column.
    if amount_column and category_column:
        category_data = (
            df.groupby(category_column)[amount_column]
            .sum()
            .sort_values(ascending=False)
            .head(10)
        )

        if not category_data.empty:
            plt.figure(figsize=(10, 5))
            category_data.plot(kind="bar")
            plt.title(f"Top Categories by {amount_column}")
            plt.xlabel(category_column)
            plt.ylabel(f"Total {amount_column}")
            plt.xticks(rotation=45)
            plt.tight_layout()

            chart_path = chart_folder / f"top_categories_by_{amount_column}.png"
            plt.savefig(chart_path)
            plt.close()

            created_charts.append(chart_path)

    return created_charts


def main():
    """
    Program starting point.
    """
    print("CSV Cleaner & Report Generator")
    print("-" * 35)

    file_path = input("Enter the path to your CSV file: ").strip()
    file_path = Path(file_path)

    if not file_path.exists():
        print(f"Error: file not found: {file_path}")
        return

    if file_path.suffix.lower() != ".csv":
        print("Error: please provide a .csv file")
        return

    output_folder = Path("output")
    output_folder.mkdir(exist_ok=True)

    print("\nReading CSV file...")
    raw_df = read_csv_safely(file_path)

    print("Cleaning data...")
    cleaned_df, cleaning_log, detected_columns = clean_dataframe(raw_df)

    cleaned_data_path = output_folder / "cleaned_data.csv"
    cleaned_df.to_csv(cleaned_data_path, index=False)

    print("Creating report...")
    report_path = create_report(
        cleaned_df,
        cleaning_log,
        detected_columns,
        output_folder,
    )

    print("Creating extra tables...")
    create_extra_tables(
        cleaned_df,
        detected_columns,
        output_folder,
    )

    print("Creating charts...")
    created_charts = create_charts(
        cleaned_df,
        detected_columns,
        output_folder,
    )

    print("\nDone!")
    print(f"Cleaned data saved to: {cleaned_data_path}")
    print(f"Main report saved to: {report_path}")

    if created_charts:
        print("\nCharts created:")
        for chart in created_charts:
            print(f"- {chart}")
    else:
        print("\nNo charts were created because the file did not contain enough suitable data.")

    print("\nDetected columns:")
    print(f"Date columns: {detected_columns['date_columns']}")
    print(f"Numeric columns: {detected_columns['numeric_columns']}")
    print(f"Category columns: {detected_columns['category_columns']}")


if __name__ == "__main__":
    main()

CSV Cleaner & Report Generator
-----------------------------------
Enter the path to your CSV file: /content/orders.csv

Reading CSV file...
Cleaning data...
Creating report...
Creating extra tables...
Creating charts...

Done!
Cleaned data saved to: output/cleaned_data.csv
Main report saved to: output/report.csv

Charts created:
- output/charts/missing_values.png
- output/charts/distribution_unit_price.png
- output/charts/monthly_total_unit_price.png
- output/charts/top_categories_by_unit_price.png

Detected columns:
Date columns: ['order_date']
Numeric columns: ['order_id', 'quantity', 'unit_price', 'discount']
Category columns: ['customer_name', 'country', 'category', 'product', 'status', 'payment_method']


In [ ]:
"""
csv_cleaner_improved.py — drop-in replacement for csv_cleaner.py

Changes vs original are marked with  # CHANGED: <reason>
"""

from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt


DATE_KEYWORDS = [
    "date", "time", "created", "ordered", "order_date",
    "invoice_date", "purchase_date", "transaction_date",
]

AMOUNT_KEYWORDS = [
    "amount", "price", "total", "revenue", "sales",
    "cost", "expense", "value", "payment", "subtotal", "profit",
]

CATEGORY_KEYWORDS = [
    "category", "type", "status", "segment", "product",
    "department", "country", "city", "region", "customer",
]


def clean_column_name(column_name):
    """
    Convert column names into clean snake_case format.

    Example:
    "Order Date " -> "order_date"
    "Customer-Name" -> "customer_name"
    """
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = column_name.strip("_")

    if column_name == "":
        return "unknown_column"

    return column_name


def make_unique_column_names(columns):
    """
    Avoid duplicated column names after cleaning.

    Example:
    ["total price", "total-price"] -> ["total_price", "total_price_2"]
    """
    used_names = {}
    unique_columns = []

    for column in columns:
        clean_name = clean_column_name(column)

        if clean_name not in used_names:
            used_names[clean_name] = 1
            unique_columns.append(clean_name)
        else:
            used_names[clean_name] += 1
            unique_columns.append(f"{clean_name}_{used_names[clean_name]}")

    return unique_columns


def read_csv_safely(file_path):
    """
    Read a CSV file, trying UTF-8 first then latin1.
    """
    try:
        return pd.read_csv(file_path)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding="latin1")


def convert_text_to_number(value):
    """
    Convert messy money/number text into a real number.

    Examples:
    "€1,200.50" -> 1200.50
    "1.200,50"  -> 1200.50
    " 450 "     -> 450.0
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if value == "":
        return pd.NA

    # Remove currency symbols, letters, and spaces.
    value = re.sub(r"[^\d,.\-]", "", value)

    # CHANGED: guard against strings that are nothing but punctuation after
    # stripping (e.g. "-", ",.") — float() would raise ValueError on these.
    if not re.search(r"\d", value):
        return pd.NA

    if "," in value and "." in value:
        if value.rfind(",") > value.rfind("."):
            # European style: 1.200,50
            value = value.replace(".", "").replace(",", ".")
        else:
            # US/UK style: 1,200.50
            value = value.replace(",", "")

    elif "," in value:
        if re.search(r",\d{1,2}$", value):
            value = value.replace(",", ".")
        else:
            value = value.replace(",", "")

    try:
        return float(value)
    except ValueError:
        return pd.NA


def parse_dates_safely(series, dayfirst=False):
    """
    Convert a column into dates.

    Parameters
    ----------
    dayfirst : bool
        Set True for European-formatted CSVs (DD/MM/YYYY).
        Defaults to False (ISO / US format) to avoid silent misparses.
        # CHANGED: was hardcoded True, which silently misparses US dates.
    """
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=dayfirst,
        format="mixed",
    )


def detect_numeric_columns(df, minimum_success_rate=0.70):
    """
    Detect columns that can probably be treated as numbers.
    Returns a dict mapping column name -> converted Series to avoid
    re-computing during clean_dataframe.
    # CHANGED: return converted series so clean_dataframe doesn't re-apply
    """
    # CHANGED: return dict {col: converted_series} to avoid double conversion
    numeric_columns = {}

    for column in df.columns:
        converted = df[column].apply(convert_text_to_number)
        success_rate = converted.notna().mean()

        name_suggests_number = any(keyword in column for keyword in AMOUNT_KEYWORDS)

        if success_rate >= minimum_success_rate or (
            name_suggests_number and success_rate >= 0.40
        ):
            numeric_columns[column] = converted

    return numeric_columns


def detect_date_columns(df, minimum_success_rate=0.70, dayfirst=False):
    """
    Detect columns that can probably be treated as dates.
    # CHANGED: accepts dayfirst so it matches parse_dates_safely's default
    """
    date_columns = []

    for column in df.columns:
        name_suggests_date = any(keyword in column for keyword in DATE_KEYWORDS)

        if pd.api.types.is_numeric_dtype(df[column]):
            continue

        parsed_dates = parse_dates_safely(df[column], dayfirst=dayfirst)
        success_rate = parsed_dates.notna().mean()

        if success_rate >= minimum_success_rate or (
            name_suggests_date and success_rate >= 0.40
        ):
            date_columns.append(column)

    return date_columns


def choose_main_column(columns, keywords):
    """
    Choose the most relevant column from a list based on keyword matching.

    Returns the first column whose name contains any of the keywords,
    or the first column if no match, or None if the list is empty.
    """
    # CHANGED: iterate columns first, then keywords so that a column
    # matching an early keyword in its name wins, regardless of keyword order.
    # Original iterated keywords-first, making keyword list order implicit priority.
    for column in columns:
        for keyword in keywords:
            if keyword in column:
                return column

    if columns:
        return columns[0]

    return None


def detect_category_columns(df, max_unique_values=30):
    """
    Detect useful category columns (low-cardinality text columns).
    """
    category_columns = []

    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            continue

        unique_count = df[column].nunique(dropna=True)
        row_count = len(df)

        if row_count == 0:
            continue

        unique_ratio = unique_count / row_count
        name_suggests_category = any(keyword in column for keyword in CATEGORY_KEYWORDS)

        if name_suggests_category or (
            2 <= unique_count <= max_unique_values and unique_ratio <= 0.50
        ):
            category_columns.append(column)

    return category_columns


def clean_dataframe(df, dayfirst=False):
    """
    Main cleaning function.

    Returns:
    1. cleaned DataFrame
    2. cleaning log with key metrics
    3. detected column groups

    Parameters
    ----------
    dayfirst : bool
        Pass True for European date formats (DD/MM/YYYY).
        # CHANGED: expose dayfirst so callers can control date parsing.
    """
    cleaning_log = {}

    cleaning_log["original_rows"] = len(df)
    cleaning_log["original_columns"] = len(df.columns)

    df = df.copy()

    df.columns = make_unique_column_names(df.columns)

    before_empty_drop = len(df)
    df = df.dropna(how="all")
    cleaning_log["empty_rows_removed"] = before_empty_drop - len(df)

    text_columns = df.select_dtypes(include=["object"]).columns

    for column in text_columns:
        df[column] = df[column].astype(str).str.strip()
        df[column] = df[column].replace(
            {"": pd.NA, "nan": pd.NA, "None": pd.NA,
             "NONE": pd.NA, "null": pd.NA, "NULL": pd.NA}
        )

    before_duplicates = len(df)
    df = df.drop_duplicates(ignore_index=True)
    cleaning_log["duplicate_rows_removed"] = before_duplicates - len(df)

    date_columns = detect_date_columns(df, dayfirst=dayfirst)

    for column in date_columns:
        df[column] = parse_dates_safely(df[column], dayfirst=dayfirst)

    # CHANGED: detect_numeric_columns now returns {col: converted_series}
    # so we reuse the already-computed result instead of applying twice.
    numeric_column_map = detect_numeric_columns(df)

    for column, converted in numeric_column_map.items():
        df[column] = pd.to_numeric(converted, errors="coerce")

    category_columns = detect_category_columns(df)

    cleaning_log["cleaned_rows"] = len(df)
    cleaning_log["cleaned_columns"] = len(df.columns)
    cleaning_log["total_missing_cells"] = int(df.isna().sum().sum())

    detected_columns = {
        "date_columns": date_columns,
        "numeric_columns": list(numeric_column_map.keys()),  # CHANGED: extract keys
        "category_columns": category_columns,
    }

    return df, cleaning_log, detected_columns


def create_extra_tables(df, detected_columns, output_folder):
    """
    Create monthly_report.csv and category_report.csv if suitable columns exist.
    """
    amount_column = choose_main_column(detected_columns["numeric_columns"], AMOUNT_KEYWORDS)
    date_column = choose_main_column(detected_columns["date_columns"], DATE_KEYWORDS)
    category_column = choose_main_column(detected_columns["category_columns"], CATEGORY_KEYWORDS)

    if amount_column and date_column:
        clean = df.dropna(subset=[date_column])

        # CHANGED: use lambda on the already-filtered frame, not the outer df.
        # Original: .assign(month=df[date_column].dt...) read from outer scope,
        # so index mismatch would silently produce NaN-filled months.
        monthly_report = (
            clean.assign(month=lambda x: x[date_column].dt.to_period("M").astype(str))
            .groupby("month", as_index=False)[amount_column]
            .sum()
            .rename(columns={amount_column: f"total_{amount_column}"})
        )

        monthly_report.to_csv(output_folder / "monthly_report.csv", index=False)

    if amount_column and category_column:
        category_report = (
            df.groupby(category_column, dropna=False)[amount_column]
            .sum()
            .sort_values(ascending=False)
            .reset_index()
            .rename(columns={amount_column: f"total_{amount_column}"})
        )

        category_report.to_csv(output_folder / "category_report.csv", index=False)


def create_report(df, cleaning_log, detected_columns, output_folder):
    """
    Create report.csv with key metrics (metric | value structure).
    """
    report_rows = []

    for metric, value in cleaning_log.items():
        report_rows.append({"metric": metric, "value": value})

    report_rows.append({
        "metric": "detected_date_columns",
        "value": ", ".join(detected_columns["date_columns"]) or "None",
    })
    report_rows.append({
        "metric": "detected_numeric_columns",
        "value": ", ".join(detected_columns["numeric_columns"]) or "None",
    })
    report_rows.append({
        "metric": "detected_category_columns",
        "value": ", ".join(detected_columns["category_columns"]) or "None",
    })

    for column in df.columns:
        report_rows.append({
            "metric": f"missing_values__{column}",
            "value": int(df[column].isna().sum()),
        })

    for column in detected_columns["numeric_columns"]:
        report_rows.extend([
            {"metric": f"sum__{column}", "value": round(df[column].sum(), 2)},
            {"metric": f"average__{column}", "value": round(df[column].mean(), 2)},
            {"metric": f"min__{column}", "value": round(df[column].min(), 2)},
            {"metric": f"max__{column}", "value": round(df[column].max(), 2)},
        ])

    for column in detected_columns["date_columns"]:
        report_rows.extend([
            {"metric": f"first_date__{column}", "value": df[column].min()},
            {"metric": f"last_date__{column}", "value": df[column].max()},
        ])

    for column in detected_columns["category_columns"]:
        top_values = df[column].value_counts(dropna=True)
        if not top_values.empty:
            report_rows.extend([
                {"metric": f"unique_values__{column}", "value": int(df[column].nunique(dropna=True))},
                {"metric": f"top_value__{column}", "value": top_values.index[0]},
                {"metric": f"top_value_count__{column}", "value": int(top_values.iloc[0])},
            ])

    report_df = pd.DataFrame(report_rows)
    report_path = output_folder / "report.csv"
    report_df.to_csv(report_path, index=False)

    return report_path


def create_charts(df, detected_columns, output_folder):
    """
    Create simple diagnostic charts saved as PNG files.
    """
    chart_folder = output_folder / "charts"
    chart_folder.mkdir(parents=True, exist_ok=True)
    created_charts = []

    missing_values = df.isna().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

    if not missing_values.empty:
        plt.figure(figsize=(10, 5))
        missing_values.plot(kind="bar")
        plt.title("Missing Values by Column")
        plt.xlabel("Column")
        plt.ylabel("Missing values")
        plt.tight_layout()
        chart_path = chart_folder / "missing_values.png"
        plt.savefig(chart_path)
        plt.close()
        created_charts.append(chart_path)

    amount_column = choose_main_column(detected_columns["numeric_columns"], AMOUNT_KEYWORDS)
    date_column = choose_main_column(detected_columns["date_columns"], DATE_KEYWORDS)
    category_column = choose_main_column(detected_columns["category_columns"], CATEGORY_KEYWORDS)

    if amount_column:
        plt.figure(figsize=(8, 5))
        df[amount_column].dropna().plot(kind="hist", bins=20)
        plt.title(f"Distribution of {amount_column}")
        plt.xlabel(amount_column)
        plt.ylabel("Frequency")
        plt.tight_layout()
        chart_path = chart_folder / f"distribution_{amount_column}.png"
        plt.savefig(chart_path)
        plt.close()
        created_charts.append(chart_path)

    if amount_column and date_column:
        monthly_data = (
            df.dropna(subset=[date_column])
            # CHANGED: use lambda to avoid outer-scope reference (same fix as create_extra_tables)
            .assign(month=lambda x: x[date_column].dt.to_period("M").astype(str))
            .groupby("month")[amount_column]
            .sum()
        )

        if not monthly_data.empty:
            plt.figure(figsize=(10, 5))
            monthly_data.plot(kind="line", marker="o")
            plt.title(f"Monthly Total: {amount_column}")
            plt.xlabel("Month")
            plt.ylabel(f"Total {amount_column}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            chart_path = chart_folder / f"monthly_total_{amount_column}.png"
            plt.savefig(chart_path)
            plt.close()
            created_charts.append(chart_path)

    if amount_column and category_column:
        category_data = (
            df.groupby(category_column)[amount_column]
            .sum()
            .sort_values(ascending=False)
            .head(10)
        )

        if not category_data.empty:
            plt.figure(figsize=(10, 5))
            category_data.plot(kind="bar")
            plt.title(f"Top Categories by {amount_column}")
            plt.xlabel(category_column)
            plt.ylabel(f"Total {amount_column}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            chart_path = chart_folder / f"top_categories_by_{amount_column}.png"
            plt.savefig(chart_path)
            plt.close()
            created_charts.append(chart_path)

    return created_charts


def main():
    print("CSV Cleaner & Report Generator")
    print("-" * 35)

    file_path = input("Enter the path to your CSV file: ").strip()
    file_path = Path(file_path)

    if not file_path.exists():
        print(f"Error: file not found: {file_path}")
        return

    if file_path.suffix.lower() != ".csv":
        print("Error: please provide a .csv file")
        return

    # CHANGED: ask the user about date format so dayfirst isn't silently assumed.
    dayfirst_input = input("Are dates in DD/MM/YYYY format? (y/N): ").strip().lower()
    dayfirst = dayfirst_input == "y"

    output_folder = Path("output")
    output_folder.mkdir(exist_ok=True)

    print("\nReading CSV file...")
    raw_df = read_csv_safely(file_path)

    print("Cleaning data...")
    cleaned_df, cleaning_log, detected_columns = clean_dataframe(raw_df, dayfirst=dayfirst)

    cleaned_data_path = output_folder / "cleaned_data.csv"
    cleaned_df.to_csv(cleaned_data_path, index=False)

    print("Creating report...")
    report_path = create_report(cleaned_df, cleaning_log, detected_columns, output_folder)

    print("Creating extra tables...")
    create_extra_tables(cleaned_df, detected_columns, output_folder)

    print("Creating charts...")
    created_charts = create_charts(cleaned_df, detected_columns, output_folder)

    print("\nDone!")
    print(f"Cleaned data saved to: {cleaned_data_path}")
    print(f"Main report saved to: {report_path}")

    if created_charts:
        print("\nCharts created:")
        for chart in created_charts:
            print(f"  - {chart}")
    else:
        print("\nNo charts created — not enough suitable data.")

    print("\nDetected columns:")
    print(f"  Date columns:     {detected_columns['date_columns']}")
    print(f"  Numeric columns:  {detected_columns['numeric_columns']}")
    print(f"  Category columns: {detected_columns['category_columns']}")


if __name__ == "__main__":
    main()

CSV Cleaner & Report Generator
-----------------------------------
Enter the path to your CSV file: /content/orders.csv
Are dates in DD/MM/YYYY format? (y/N): y

Reading CSV file...
Cleaning data...
Creating report...
Creating extra tables...
Creating charts...

Done!
Cleaned data saved to: output/cleaned_data.csv
Main report saved to: output/report.csv

Charts created:
  - output/charts/missing_values.png
  - output/charts/distribution_unit_price.png
  - output/charts/monthly_total_unit_price.png
  - output/charts/top_categories_by_unit_price.png

Detected columns:
  Date columns:     ['order_date']
  Numeric columns:  ['order_id', 'quantity', 'unit_price', 'discount']
  Category columns: ['customer_name', 'country', 'category', 'product', 'status', 'payment_method']
